# χ4 Feasibility Probe — dynamic heterogeneity in iSCAT

**Standalone.** Computes the four-point dynamic susceptibility χ4(τ) for the whole cell
and checks whether it has a peak at a characteristic time τ* — the signature of
**dynamic heterogeneity** (cooperative fast/slow switching, e.g. LLPS), which the ACF /
(γ,α) cannot see.

χ4(τ) = N·Var_t[Q(t,τ)],  Q = spatial-mean overlap of z-scored fluctuations (common-mode
global flicker removed). A TIME-SHUFFLE control gives the noise floor: real heterogeneity
⇒ χ4(τ*) ≫ shuffled. Outcomes: **peak** (worth pursuing) / **no peak** / **SNR-starved**
(honest degradation — like STICS sub-PSF).


In [ ]:
# ── setup: clone repo (for utils/gpu_chi4.py) + deps ─────────────────────
import os, subprocess, sys
REPO='https://github.com/breezy90126/iscors-net.git'
BRANCH='claude/brave-ramanujan-33eps3'
REPO_DIR='/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin'],check=False)
    subprocess.run(['git','-C',REPO_DIR,'checkout',BRANCH],check=False)
    subprocess.run(['git','-C',REPO_DIR,'pull','origin',BRANCH],check=False)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO,REPO_DIR],check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip','install','-q','tifffile','scipy'],check=False)
print('setup done:', os.getcwd())


In [ ]:
# ── config (edit paths to your data) ─────────────────────────────────────
import numpy as np
VIDEO_PATH = '/content/drive/MyDrive/iscors_test/large_file.zip'  # zip or .tif
VIDEO_FNAME= 'COBRI_rarw_video.tif'   # name inside the zip (ignored if VIDEO_PATH is .tif)
N_FRAMES   = 2000          # χ4 wants as many frames as possible
BIN_FACTOR = 2
MIN_CV     = 0.005
TAUS       = (1,2,3,4,6,8,12,16,24,32,48,64,96,128,192,256)  # denser τ → cleaner peak
SAVE_DIR   = '/content/drive/MyDrive/iscors_test'
os.makedirs(SAVE_DIR, exist_ok=True)
print('TAUS =', TAUS)


In [ ]:
# ── load + minimal preprocess (bin only; χ4 removes static + common-mode itself) ──
import tifffile, zipfile, io
def _load(path, fname, n):
    if path.endswith('.zip'):
        with zipfile.ZipFile(path) as z:
            with z.open(fname) as f: arr = tifffile.imread(io.BytesIO(f.read()))
    else:
        arr = tifffile.imread(path)
    return arr[:n].astype(np.float32)
vid = _load(VIDEO_PATH, VIDEO_FNAME, N_FRAMES)
T,H,W = vid.shape
if BIN_FACTOR>1:
    Hb,Wb = H//BIN_FACTOR, W//BIN_FACTOR
    vid = vid[:, :Hb*BIN_FACTOR, :Wb*BIN_FACTOR].reshape(T,Hb,BIN_FACTOR,Wb,BIN_FACTOR).mean((2,4))
print('video:', vid.shape)


In [ ]:
# ── compute χ4(τ) + time-shuffle control + verdict ───────────────────────
import importlib, utils.gpu_chi4 as _c4; importlib.reload(_c4)
from utils.gpu_chi4 import chi4_probe
out = chi4_probe(vid, TAUS, min_cv=MIN_CV, verbose=True)
print()
print('τ      :', out['taus'].astype(int))
print('χ4     :', np.round(out['chi4'],1))
print('χ4 shuf:', np.round(out['chi4_shuffled'],1))
print('relax  :', np.round(out['relax'],3), ' (should decay with τ)')


In [ ]:
# ── plot + record ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
ax[0].semilogx(out['taus'], out['chi4'], 'o-', label='χ4 (real)')
ax[0].semilogx(out['taus'], out['chi4_shuffled'], 's--', color='gray', label='χ4 (time-shuffled = noise floor)')
if out['resolved']:
    ax[0].axvline(out['tau_star'], color='r', ls=':', label=f"τ*={out['tau_star']:.0f}")
ax[0].set_xlabel('τ (lag)'); ax[0].set_ylabel('χ4(τ)'); ax[0].legend(fontsize=8)
ax[0].set_title(f"χ4  (N={out['N']} px)  peak={out['peak']:.1f}  {out['snr']:.1f}× floor")
ax[1].semilogx(out['taus'], out['relax'], 'o-')
ax[1].set_xlabel('τ (lag)'); ax[1].set_ylabel('⟨Q(τ)⟩ (relaxation)')
ax[1].set_title('overlap relaxation (sanity: should decay)')
verdict = ('RESOLVED: dynamic heterogeneity, τ*=%.0f → χ4 worth pursuing'%out['tau_star']
           if out['resolved'] else
           'NOT resolved: no peak above noise floor → homogeneous/static/SNR-starved')
fig.suptitle('χ4 feasibility probe — '+verdict, fontsize=12)
plt.tight_layout(); plt.show()
fig.savefig(os.path.join(SAVE_DIR,'chi4_probe.png'), dpi=120, bbox_inches='tight')
with open(os.path.join(SAVE_DIR,'chi4_probe.txt'),'w') as f:
    f.write('=== chi4 feasibility probe ===\n')
    f.write(f"N_cell={out['N']}  peak_chi4={out['peak']:.2f}  shuffle_floor={out['shuffle_floor']:.2f}  snr={out['snr']:.2f}\n")
    f.write(f"tau_star={out['tau_star']}  resolved={out['resolved']}\n")
    f.write('VERDICT: '+verdict+'\n')
print('VERDICT:', verdict)
print('saved →', SAVE_DIR)


## How to read it
- **Clear peak at τ*, well above the shuffled (gray) floor** → dynamic heterogeneity is
  real and resolvable; τ* is its timescale, peak height ≈ size of cooperative regions.
  χ4 is worth developing (incl. an ML estimator, since classical χ4 is noise-starved).
- **χ4 ≈ shuffled floor / no peak** → no resolvable dynamic heterogeneity (homogeneous,
  or purely static differences, or SNR-starved). Honest degradation — like STICS sub-PSF.
- **`relax` not decaying** → preprocessing problem (check common-mode / drift) before trusting χ4.
